In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
import pickle
import seaborn as sns
from lifelines import KaplanMeierFitter
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from scipy.stats import norm, logistic, gumbel_r
from sklearn.calibration import calibration_curve
from sklearn.metrics import confusion_matrix
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

pd.options.display.float_format = "{:.5f}".format

# Extra data cleaning

Faults cleaned before publishing of the datasets

## Fixing indexing in deciduous dogs

indexing issues where some dogs have same ID --> resetting these matches the indexes used in the teeth sheet

In [2]:
deciduous_dogs = pd.read_excel(
    "../data/deciduous_EDA_clean.xlsx", index_col=0, sheet_name="dogs"
)

In [3]:
deciduous_dogs.index.value_counts(sort=True)

DOG
291     2
292     2
293     2
294     2
295     2
       ..
997     1
998     1
999     1
1000    1
6       1
Name: count, Length: 982, dtype: int64

In [4]:
deciduous_dogs = deciduous_dogs.reset_index(drop=True)
deciduous_dogs = deciduous_dogs.reset_index(drop=False, names="DOG")
deciduous_dogs["DOG"] = deciduous_dogs["DOG"] + 1
deciduous_dogs.set_index("DOG", inplace=True)
deciduous_teeth = pd.read_excel(
    "../data/deciduous_EDA_clean.xlsx", index_col=0, sheet_name="teeth"
)

deciduous_teeth = deciduous_teeth.join(deciduous_dogs, how="inner")

deciduous_teeth.reset_index(inplace=True)

In [5]:
deciduous_teeth["DOG"].value_counts(sort=True)

DOG
1001    56
1       56
2       56
3       56
4       56
        ..
14      56
13      56
12      56
11      56
10      56
Name: count, Length: 1001, dtype: int64

## Fix switched values

Values of stage 1 that should be those of stage 2 and other way around

- Hond 170 voor tanden 506, 507, 606, 607, 706, 707, 806, 807 waarden omgedraaid
- Hond 172 voor tanden 508, 608 waarden omgedraaid
- Hond 213 voor tanden 503, 603 waarden omgedraaid
- Hond 214 voor tanden 703, 803 waarden omgedraaid
- Hond 448 voor tanden 503, 603 waarden omgedraaid
- Hond 773 voor tanden 502, 503, 602, 603 waarden omgedraaid
- Hond 780 voor tanden 706, 806 waarden omgedraaid
- Hond 991 voor tanden 701, 702, 703, 801, 802, 803 waarden omgedraaid

In [6]:
def fix_order(df_in, dog: int, teeth: list):
    teeth_df = df_in.copy()

    fault = (teeth_df["DOG"] == dog) & (teeth_df["TOOTH NUMBER"].isin(teeth))

    teeth_df.loc[fault, ["LOWER", "UPPER"]] = (
        teeth_df.loc[fault]
        .sort_values("DEVELOPM STAGE")
        .groupby("TOOTH NUMBER")[["LOWER", "UPPER"]]
        .transform(lambda df: df.iloc[::-1].values)
    )

    return teeth_df

In [7]:
deciduous_teeth = fix_order(
    deciduous_teeth, 170, [506, 507, 606, 607, 706, 707, 806, 807]
)

deciduous_teeth = fix_order(deciduous_teeth, 172, [508, 608])

deciduous_teeth = fix_order(deciduous_teeth, 213, [503, 603])

deciduous_teeth = fix_order(deciduous_teeth, 214, [703, 803])

deciduous_teeth = fix_order(deciduous_teeth, 448, [503, 603])

deciduous_teeth = fix_order(deciduous_teeth, 773, [502, 503, 602, 603])

deciduous_teeth = fix_order(deciduous_teeth, 780, [706, 806])

deciduous_teeth = fix_order(deciduous_teeth, 991, [701, 702, 703, 801, 802, 803])

## Fix typo's

- Hond 809 voor tanden 508 Dit klopt niet: stadium 2 moest 43-44 zijn ipv 33-34

In [8]:
deciduous_teeth.loc[
    (deciduous_teeth["DOG"] == 809)
    & (deciduous_teeth["TOOTH NUMBER"] == 508)
    & (deciduous_teeth["DEVELOPM STAGE"] == 2),
    ["LOWER", "UPPER"],
] = [43, 44]

## Fill in NaNs where possible

- UPPER waarde van stage 2 overnemen en ook als UPPER waarde van stage 1 zetten
- LOWER waarde van stage 1 overnemen en ook als LOWER waarde van stage 2 zetten

==> Het klopt inderdaad dat, als een tand voor de leeftijd van bijvoorbeeld 63 dagen volledig is doorgekomen (stadium 2) die tand voor die leeftijd ook al gedeeltelijk is doorgekomen (stadium 1). En omgekeerd! Dus dit kan je inderdaad zo aanpassen

In [9]:
# groupby dog and tooth --> bfill = fill in NaN of previous is next is filled in (eg: fill in stage 1 UPPER if stage 2 UPPER is known)
deciduous_teeth["UPPER"] = deciduous_teeth.groupby(["DOG", "TOOTH NUMBER"])[
    "UPPER"
].bfill()

# groupby dog and tooth --> ffill = fill in NaN of next is previous is filled in (eg: fill in stage 2 LOWER if stage 1 LOWER is known)
deciduous_teeth["LOWER"] = deciduous_teeth.groupby(["DOG", "TOOTH NUMBER"])[
    "LOWER"
].ffill()

### Fix SENS values

- 0 = gn censoring
- 1 = geen info
- 2 = left
- 3 = interval
- 4 = right
- 5 = right of non-responder

In [10]:
deciduous_teeth.loc[
    (deciduous_teeth["LOWER"].isna() & deciduous_teeth["UPPER"].isna()), "SENS"
] = 1

deciduous_teeth.loc[
    (deciduous_teeth["LOWER"].notna() & deciduous_teeth["UPPER"].notna()), "SENS"
] = 3

deciduous_teeth.loc[
    (
        (deciduous_teeth["LOWER"] == deciduous_teeth["UPPER"])
        & (deciduous_teeth["LOWER"].notna()),
        "SENS",
    )
] = 0

deciduous_teeth.loc[
    (deciduous_teeth["LOWER"].isna() & deciduous_teeth["UPPER"].notna()), "SENS"
] = 2

deciduous_teeth.loc[
    (deciduous_teeth["LOWER"].notna() & deciduous_teeth["UPPER"].isna()), "SENS"
] = 4

In [11]:
deciduous_teeth["SENS"].value_counts()

SENS
2    29430
3    13950
4     7228
1     3004
0     2444
Name: count, dtype: int64

## Save cleaned dataset

In [15]:
deciduous_teeth.reset_index(inplace=True, drop=True)

In [16]:
deciduous_teeth.to_csv(
    "../data/deciduous_teeth_fully_cleaned.csv", sep=";", index="DOG"
)

## Fix switched values in permanent teeth

In [17]:
permanent_dogs = pd.read_excel(
    "../data/permanent_EDA_clean.xlsx", index_col=0, sheet_name="dogs"
)
permanent_teeth = pd.read_excel(
    "../data/permanent_EDA_clean.xlsx", index_col=0, sheet_name="teeth"
)

permanent_teeth = permanent_teeth.join(permanent_dogs, how="inner")

permanent_teeth = permanent_teeth.reset_index()

In [18]:
permanent_teeth["DOG"].value_counts()

DOG
346    130
9      112
440    112
1      112
2      112
      ... 
21     112
22     112
23     112
425    112
350     94
Name: count, Length: 440, dtype: int64

In [19]:
# fix wrongly assigned cols: last vals of 346 should be for 350
mask = (permanent_teeth["DOG"] == 346) & (
    permanent_teeth["TOOTH NUMBER"].isin([201, 101, 301, 401, 202, 102])
)
positions = np.flatnonzero(mask)
positions_to_change = positions[18:]
permanent_teeth.loc[positions_to_change, "DOG"] = 350

permanent_teeth.reset_index(inplace=True)
# fix order issues
permanent_teeth["TOOTH NUMBER"] = pd.Categorical(
    permanent_teeth["TOOTH NUMBER"], permanent_teeth["TOOTH NUMBER"].unique()
)
permanent_teeth = permanent_teeth.sort_values(
    ["DOG", "TOOTH NUMBER", "DEVELOPM STAGE"], ignore_index=True
)

## Fix mistakes

- Hond 203 tanden 310, 410 waarden omgedraaid
- Hond 353 tanden 403 Dit klopt niet: stadium 3 moest ‘150’ zijn en geen ‘143’

In [20]:
permanent_teeth[
    (permanent_teeth["DOG"] == 203) & (permanent_teeth["TOOTH NUMBER"] == 310)
]

,index,DOG,TOOTH NUMBER,MAX/MAND,SIDE (L/R),POSITION,DEVELOPM STAGE,LOWER,UPPER,SENS,SEX,BREED,LITTER,BREED SIZE,SKULL TYPE,BREED CLADE,RELIABILITY (1/2),STUDY
22728,22728,203,310,MAND,L,10,2,157.00000,NaN,4,M,Golden Retriever,Single,Large,Meso,Q,1,LONG
22729,22729,203,310,MAND,L,10,3,154.00000,154.00000,0,M,Golden Retriever,Single,Large,Meso,Q,1,LONG


In [21]:
permanent_teeth.loc[
    (permanent_teeth["DOG"] == 203)
    & (permanent_teeth["TOOTH NUMBER"].isin([310, 410]))
    & (permanent_teeth["DEVELOPM STAGE"] == 2),
    ["LOWER", "UPPER"],
] = [154, 154]

permanent_teeth.loc[
    (permanent_teeth["DOG"] == 203)
    & (permanent_teeth["TOOTH NUMBER"].isin([310, 410]))
    & (permanent_teeth["DEVELOPM STAGE"] == 3),
    ["LOWER", "UPPER"],
] = [157, np.nan]

In [22]:
permanent_teeth.loc[
    (permanent_teeth["DOG"] == 353)
    & (permanent_teeth["TOOTH NUMBER"] == 403)
    & (permanent_teeth["DEVELOPM STAGE"] == 3),
    ["LOWER", "UPPER"],
] = [150, 150]

## Fill in NaNs permanent

only on stage 2 and 3, bc stage 1 can be bigger than stage 2 and 3

In [23]:
mask = permanent_teeth["DEVELOPM STAGE"].isin([2, 3])

permanent_teeth.loc[mask, "UPPER"] = (
    permanent_teeth.loc[mask].groupby(["DOG", "TOOTH NUMBER"])["UPPER"].bfill()
)

permanent_teeth.loc[mask, "LOWER"] = (
    permanent_teeth.loc[mask].groupby(["DOG", "TOOTH NUMBER"])["LOWER"].ffill()
)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_22656\616945962.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  permanent_teeth.loc[mask].groupby(["DOG", "TOOTH NUMBER"])["UPPER"].bfill()
C:\Users\Administrator\AppData\Local\Temp\ipykernel_22656\616945962.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  permanent_teeth.loc[mask].groupby(["DOG", "TOOTH NUMBER"])["LOWER"].ffill()


- 0 = gn censoring
- 1 = geen info
- 2 = left
- 3 = interval
- 4 = right
- 5 = right of non-responder

In [24]:
permanent_teeth.loc[
    (permanent_teeth["LOWER"].isna() & permanent_teeth["UPPER"].isna()), "SENS"
] = 1

permanent_teeth.loc[
    (permanent_teeth["LOWER"].notna() & permanent_teeth["UPPER"].notna()), "SENS"
] = 3

permanent_teeth.loc[
    (
        (permanent_teeth["LOWER"] == permanent_teeth["UPPER"])
        & (permanent_teeth["LOWER"].notna()),
        "SENS",
    )
] = 0

permanent_teeth.loc[
    (permanent_teeth["LOWER"].isna() & permanent_teeth["UPPER"].notna()), "SENS"
] = 2

permanent_teeth.loc[
    (permanent_teeth["LOWER"].notna() & permanent_teeth["UPPER"].isna()), "SENS"
] = 4

In [25]:
permanent_teeth["SENS"].value_counts()

SENS
1    14198
3    12540
0     9797
4     8718
2     4027
Name: count, dtype: int64

In [261]:
permanent_teeth.to_csv(
    "../data/permanent_teeth_fully_cleaned.csv", sep=";", index="DOG"
)